# Credit Card Fraud Detection

This notebook builds a classification model to detect fraudulent credit card transactions in a highly imbalanced dataset.

## 1. Problem Definition

In a statement,
> Given anonymized transaction data, can we identify which transactions are fraudulent?

## 2. Data

The dataset is the [Credit Card Fraud Detection dataset](https://www.kaggle.com/mlg-ulb/creditcardfraud) from the Machine Learning Group at ULB (Universite Libre de Bruxelles), available on Kaggle.

It contains 284,807 transactions made by European cardholders over two days in September 2013. Only 492 of them are fraudulent, about 0.17% of the total. Because of confidentiality, the original transaction features were transformed with PCA into 28 anonymized components (`V1` to `V28`); only `Time`, `Amount`, and `Class` (the target) keep their original meaning.

## 3. Evaluation

Because the dataset is heavily imbalanced, accuracy alone would be misleading here: a model predicting "not fraud" for every transaction would already score above 99.8% accuracy while catching zero fraud. The evaluation will focus on precision, recall, and F1-score for the fraud class, plus ROC-AUC and precision-recall AUC, which are more informative on imbalanced data.

## 4. Features

Data dictionary (from the dataset description on Kaggle):

* `Time` - seconds elapsed between this transaction and the first transaction in the dataset
* `V1` to `V28` - principal components from a PCA transformation, anonymized for confidentiality reasons (the original features and any background information about them are not available)
* `Amount` - transaction amount
* `Class` - target variable: 1 = fraud, 0 = normal transaction

## 5. Preparing the tools

We use pandas, NumPy, matplotlib and seaborn for data analysis and visualization, and scikit-learn for modelling.

In [ ]:
# Import all tools we need
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

## Data Exploration (EDA)

The goal here is to understand the data before doing any modelling:
1. How many transactions and features do we have?
2. Are there missing values?
3. How imbalanced is the target class, exactly?
4. What do `Time` and `Amount` look like for fraud vs normal transactions?

In [ ]:
# Load the dataset
df = pd.read_csv("../data/creditcard.csv")
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
# Check for missing values
df.isna().sum()

In [ ]:
df.describe()

### Class balance

In [ ]:
# How many fraud vs normal transactions do we have?
class_counts = df["Class"].value_counts()
class_pct = df["Class"].value_counts(normalize=True) * 100
print(class_counts)
print(class_pct)

In [ ]:
class_counts.plot(kind="bar", color=["lightblue", "salmon"])
plt.title("Transaction Class Distribution (0 = Normal, 1 = Fraud)")
plt.xlabel("Class")
plt.ylabel("Number of transactions")
plt.xticks(rotation=0)
plt.show()

### Amount and Time by class

`V1`-`V28` are already PCA components, roughly centered and scaled. `Amount` and `Time` are still in their original units, so they are worth checking separately.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x="Class", y="Amount", ax=axes[0])
axes[0].set_title("Transaction Amount by Class")
axes[0].set_xticklabels(["Normal", "Fraud"])

sns.boxplot(data=df, x="Class", y="Time", ax=axes[1])
axes[1].set_title("Transaction Time by Class")
axes[1].set_xticklabels(["Normal", "Fraud"])

plt.tight_layout()
plt.show()

### Correlation with the target

In [ ]:
# Correlation of each feature with the target, sorted by absolute value
corr_with_target = df.corr()["Class"].drop("Class").sort_values(key=abs, ascending=False)
corr_with_target.head(15)

In [ ]:
plt.figure(figsize=(8, 8))
corr_with_target.head(15).plot(kind="barh")
plt.title("Top 15 features by absolute correlation with Class")
plt.xlabel("Correlation with Class")
plt.gca().invert_yaxis()
plt.show()

## Preprocessing

`V1` to `V28` are already PCA-transformed and do not need scaling. `Amount` and `Time` are on very different scales (dollars vs seconds), so we scale them before modelling.

We split into train/test first, then fit the scaler on the training set only, and apply it to both sets. This avoids leaking information about the test set's distribution into the scaler.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

X = df.drop("Class", axis=1)
y = df["Class"]

# Stratify on y to keep the same fraud ratio in both train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

X_train.shape, X_test.shape, y_train.sum(), y_test.sum()

In [ ]:
scaler = RobustScaler()

# Fit on training data only, then apply the same transformation to both sets
X_train[["Amount", "Time"]] = scaler.fit_transform(X_train[["Amount", "Time"]])
X_test[["Amount", "Time"]] = scaler.transform(X_test[["Amount", "Time"]])

X_train[["Amount", "Time"]].describe()

## Evaluation function

Since the dataset is heavily imbalanced, accuracy is not a useful metric on its own. We build a function that reports precision, recall, f1-score, ROC-AUC, and PR-AUC (average precision) for the fraud class, plus a confusion matrix.

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)

def evaluate_model(model, X_test, y_test, model_name="Model", plot=True):
    """
    Evaluate a fitted classifier on the fraud detection task.
    Returns a dict of metrics and optionally plots ROC and Precision-Recall curves.
    """
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    print(f"--- {model_name} ---")
    print(classification_report(y_test, y_pred, target_names=["Normal", "Fraud"]))

    roc_auc = roc_auc_score(y_test, y_proba)
    pr_auc = average_precision_score(y_test, y_proba)
    print(f"ROC-AUC: {roc_auc:.4f}")
    print(f"PR-AUC (average precision): {pr_auc:.4f}")

    if plot:
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        cm = confusion_matrix(y_test, y_pred)
        sns.heatmap(cm, annot=True, fmt="d", cbar=False, ax=axes[0],
                    xticklabels=["Normal", "Fraud"], yticklabels=["Normal", "Fraud"])
        axes[0].set_title(f"{model_name} - Confusion Matrix")
        axes[0].set_xlabel("Predicted")
        axes[0].set_ylabel("True")

        RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1])
        axes[1].set_title(f"{model_name} - ROC Curve")

        PrecisionRecallDisplay.from_predictions(y_test, y_proba, ax=axes[2])
        axes[2].set_title(f"{model_name} - Precision-Recall Curve")

        plt.tight_layout()
        plt.show()

    return {
        "model": model_name,
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
    }

results = []

## Baseline model (no imbalance handling)

A plain LogisticRegression on the raw class distribution, as a reference point before trying anything to handle the imbalance.

In [ ]:
from sklearn.linear_model import LogisticRegression

baseline_model = LogisticRegression(max_iter=1000, random_state=42)
baseline_model.fit(X_train, y_train)

results.append(evaluate_model(baseline_model, X_test, y_test, model_name="Baseline LogisticRegression"))

## Handling class imbalance

Two common approaches, tried separately:
1. **Class weights**: tell the model to penalize mistakes on the minority class (fraud) more heavily, without changing the training data itself.
2. **SMOTE (Synthetic Minority Oversampling Technique)**: generate synthetic fraud examples in the training set only, to rebalance the classes before fitting.

SMOTE is applied strictly on the training set, after the train/test split. Applying it before the split (or on the test set) would leak synthetic near-duplicates of test fraud cases into training, which would inflate the evaluation metrics artificially - the same kind of mistake as fitting preprocessing statistics on data that includes the validation/test set.

`imbalanced-learn` needs to be installed for the SMOTE step: `pip install imbalanced-learn`.

### Class weights

In [ ]:
from sklearn.ensemble import RandomForestClassifier

log_reg_balanced = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
log_reg_balanced.fit(X_train, y_train)
results.append(evaluate_model(log_reg_balanced, X_test, y_test, model_name="LogisticRegression (class_weight=balanced)"))

In [ ]:
rf_balanced = RandomForestClassifier(n_estimators=100, class_weight="balanced", n_jobs=-1, random_state=42)
rf_balanced.fit(X_train, y_train)
results.append(evaluate_model(rf_balanced, X_test, y_test, model_name="RandomForest (class_weight=balanced)"))

### SMOTE oversampling

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# Confirm the classes are now balanced in the training set (test set is untouched)
y_train_smote.value_counts()

In [ ]:
log_reg_smote = LogisticRegression(max_iter=1000, random_state=42)
log_reg_smote.fit(X_train_smote, y_train_smote)
results.append(evaluate_model(log_reg_smote, X_test, y_test, model_name="LogisticRegression (SMOTE)"))

In [ ]:
rf_smote = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
rf_smote.fit(X_train_smote, y_train_smote)
results.append(evaluate_model(rf_smote, X_test, y_test, model_name="RandomForest (SMOTE)"))

## Model comparison

All five models (baseline, two class-weighted, two SMOTE-based) are evaluated on the exact same, untouched test set, so their metrics are directly comparable.

In [ ]:
results_df = pd.DataFrame(results).set_index("model")
results_df.sort_values("pr_auc", ascending=False)

In [ ]:
results_df[["precision", "recall", "f1", "pr_auc"]].plot(kind="bar", figsize=(12, 6))
plt.title("Model comparison on the test set")
plt.ylabel("Score")
plt.xticks(rotation=30, ha="right")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## Feature importance

Using the RandomForest trained on the class-weighted data (no synthetic rows, so the importances reflect the original data directly).

In [ ]:
importances = pd.Series(rf_balanced.feature_importances_, index=X_train.columns).sort_values(ascending=False)
importances.head(15)

In [ ]:
plt.figure(figsize=(8, 8))
importances.head(15).plot(kind="barh")
plt.title("Top 15 feature importances (RandomForest, class_weight=balanced)")
plt.xlabel("Importance")
plt.gca().invert_yaxis()
plt.show()

## Cross-validation

A single train/test split can be misleading with so few fraud cases in the test set. Stratified k-fold cross-validation, scored on PR-AUC (more informative than accuracy on imbalanced data), gives a more robust estimate for the best-performing model found above.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Replace `rf_balanced` below with whichever model scores best in the comparison above
cv_scores = cross_val_score(rf_balanced, X, y, cv=cv, scoring="average_precision", n_jobs=-1)
print(f"Cross-validated PR-AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
cv_scores

## Notes on evaluation (to fill in after running this notebook)

This notebook has not been executed yet (no local copy of the dataset). A few things worth checking once it runs on the real data, rather than assumptions made in advance:

- **Compare recall, not just accuracy, across the five models.** The whole point of trying class weights and SMOTE is to catch more fraud cases (higher recall) without destroying precision. Whichever approach wins on PR-AUC in the comparison table is the one to write up as the main result.
- **Check the confusion matrices, not just the summary metrics.** With only ~100 fraud cases in a 20% test split, a handful of missed or extra flagged transactions can move recall/precision by several points - the raw counts in the confusion matrix give a sense of how much the metrics can be trusted.
- **RandomForest feature importances are only informative up to a point here**, since `V1`-`V28` are anonymized PCA components with no direct business meaning - useful for saying "these components matter most", not for explaining why in domain terms.
- **This dataset is from September 2013 and already anonymized/PCA-transformed by the original authors**, so there is no real preprocessing decision left to second-guess on the `V1`-`V28` columns; the meaningful methodology choices are in how the class imbalance is handled and how the model is evaluated, which is why this notebook focuses there.